Код, который использует дифференциальную геометрию для поиска аномалий на снимках. Используем топологический/геометрический анализ как признаковый слой для обучения нейросетей. Далее используем топологические/геометрические признаки вместе с обычными признаками в сегментирующей сети. В конце сравниваем результаты обычной сети и сети с таким слоем

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import ripser
import persim
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform
import gudhi as gd
from gudhi.representations import Landscape, Silhouette, PersistenceImage
import tadasets
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import networkx as nx
import nibabel as nib
import os
from tqdm import tqdm
from skimage.transform import resize
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, roc_curve
import warnings
import time
from scipy.linalg import svd
import pydicom
from pathlib import Path
from PIL import Image
import glob
from scipy.ndimage import gaussian_filter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from scipy.ndimage import gaussian_filter, binary_dilation, binary_closing, binary_fill_holes
from skimage import exposure, morphology, segmentation, measure
import torch.nn.functional as F
warnings.filterwarnings("ignore", category=UserWarning)

target_shape = (128, 128, 64)

In [3]:
def load_custom_dataset(base_path, max_samples=None):
    tumor_dir = os.path.join(base_path, "Tumor")
    mask_dir = os.path.join(base_path, "Mask")
    
    tumor_files = sorted(glob.glob(os.path.join(tumor_dir, "*.tif")))
    mask_files = sorted(glob.glob(os.path.join(mask_dir, "*.tif")))
    
    if max_samples is not None:
        tumor_files = tumor_files[:max_samples]
        mask_files = mask_files[:max_samples]
    
    tumor_images = []
    mask_images = []
    
    for tumor_file, mask_file in zip(tumor_files, mask_files):
        tumor_img = np.array(Image.open(tumor_file))
        mask_img = np.array(Image.open(mask_file))
        
        if not (np.array_equal(np.unique(mask_img), [0, 1]) or mask_img.dtype == bool):
            mask_img = mask_img > 0
        
        tumor_images.append(tumor_img)
        mask_images.append(mask_img)
    
    return np.array(tumor_images), np.array(mask_images)

# Архитектура сегментационных моделей (базовая U-Net и расширенная U-Net)

In [ ]:
def load_custom_dataset(base_path, max_samples=None):

    tumor_dir = os.path.join(base_path, "Tumor")
    mask_dir = os.path.join(base_path, "Mask")
    
    tumor_files = sorted(glob.glob(os.path.join(tumor_dir, "*.tif")))
    mask_files = sorted(glob.glob(os.path.join(mask_dir, "*.tif")))
    
    if max_samples is not None:
        tumor_files = tumor_files[:max_samples]
        mask_files = mask_files[:max_samples]
    
    tumor_images = []
    mask_images = []
    
    for tumor_file, mask_file in zip(tumor_files, mask_files):
        tumor_img = np.array(Image.open(tumor_file))
        mask_img = np.array(Image.open(mask_file))
        
        if not (np.array_equal(np.unique(mask_img), [0, 1]) or mask_img.dtype == bool):
            mask_img = mask_img > 0
        
        tumor_images.append(tumor_img)
        mask_images.append(mask_img)
    
    return np.array(tumor_images), np.array(mask_images)

def compute_fiber_bundle_features(images, patch_size=7, stride=4, n_neighbors=10):
    """
    Вычисляет признаки на основе расслоения (fiber bundle) для создания карт аномалий
    """
    all_feature_maps = []
    
    for img_idx, image in enumerate(tqdm(images, desc="Computing features")):
        h, w = image.shape[:2]
        
        # Подготовка патчей
        patches = []
        positions = []
        
        for i in range(0, h - patch_size + 1, stride):
            for j in range(0, w - patch_size + 1, stride):
                patch = image[i:i+patch_size, j:j+patch_size]
                patches.append(patch.flatten())
                positions.append((i, j))
        
        patches = np.array(patches)
        
        # Вычисляем kNN для патчей
        nbrs = NearestNeighbors(n_neighbors=n_neighbors+1).fit(patches)
        distances, indices = nbrs.kneighbors(patches)
        indices = indices[:, 1:] 
        
        # Вычисляем кривизну
        curvature_scores = np.zeros(len(patches))
        
        for i in range(len(patches)):
            neighbors = patches[indices[i]]
            center = patches[i]
            centered_neighbors = neighbors - center
            
            cov_matrix = np.dot(centered_neighbors.T, centered_neighbors)
            eigenvalues = np.linalg.eigvalsh(cov_matrix)
            eigenvalues = eigenvalues[eigenvalues > 1e-10]
            
            if len(eigenvalues) > 1:
                min_eigenvalue = np.min(eigenvalues)
                max_eigenvalue = np.max(eigenvalues)
                curvature_scores[i] = 1.0 - (min_eigenvalue / max_eigenvalue)
            else:
                curvature_scores[i] = 1.0

        if np.max(curvature_scores) > np.min(curvature_scores):
            curvature_scores = (curvature_scores - np.min(curvature_scores)) / (np.max(curvature_scores) - np.min(curvature_scores))
        
        curvature_map = np.zeros((h, w))
        count_map = np.zeros((h, w))
        
        for score, (i, j) in zip(curvature_scores, positions):
            curvature_map[i:i+patch_size, j:j+patch_size] += score
            count_map[i:i+patch_size, j:j+patch_size] += 1
        
        count_map[count_map == 0] = 1
        curvature_map = curvature_map / count_map
        
        curvature_map = gaussian_filter(curvature_map, sigma=1.0)
        
        # Нормализуем в диапазон [0, 1]
        if np.max(curvature_map) > np.min(curvature_map):
            curvature_map = (curvature_map - np.min(curvature_map)) / (np.max(curvature_map) - np.min(curvature_map))
            
        all_feature_maps.append(curvature_map)
    
    return np.array(all_feature_maps)

class DoubleConv(nn.Module):
    """(Conv -> BN -> ReLU) x 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    """Стандартная архитектура U-Net"""
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        
        # Encoder
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        
        # Bottleneck
        self.bottleneck = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))
        
        # Decoder
        stride = 2
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=stride)
        self.conv1 = DoubleConv(1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=stride)
        self.conv2 = DoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=stride)
        self.conv3 = DoubleConv(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=stride)
        self.conv4 = DoubleConv(128, 64)
        
        # Выходной слой
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)
        
    def forward(self, x):
        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.bottleneck(x4)
        
        # Decoder
        x = self.up1(x5)
        x = torch.cat([x, x4], dim=1)
        x = self.conv1(x)
        
        x = self.up2(x)
        x = torch.cat([x, x3], dim=1)
        x = self.conv2(x)
        
        x = self.up3(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv3(x)
        
        x = self.up4(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv4(x)
        
        x = self.outc(x)
        return torch.sigmoid(x)

class EnhancedUNet(nn.Module):
    """U-Net с добавлением геометрических/топологических признаков"""
    def __init__(self, in_channels=3, feature_channels=1, out_channels=1):
        super(EnhancedUNet, self).__init__()
        self.in_channels = in_channels
        self.feature_channels = feature_channels
        
        # Обработка обычного изображения
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        
        # Обработка карты признаков - обеспечиваем совпадение размерностей
        self.feat_conv1 = DoubleConv(feature_channels, 64)
        self.feat_down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.feat_down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.feat_down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))  # Добавляем еще один уровень
        
        # Bottleneck - теперь с тензорами одинаковой размерности
        self.bottleneck = nn.Sequential(nn.MaxPool2d(2), DoubleConv(1024, 1024))  # 512+512 = 1024
        
        # Decoder
        stride = 2
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=stride)
        self.conv1 = DoubleConv(1024, 512)  # 512+512
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=stride)
        self.conv2 = DoubleConv(512, 256)  # 256+256
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=stride)
        self.conv3 = DoubleConv(256, 128)  # 128+128
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=stride)
        self.conv4 = DoubleConv(128, 64)  # 64+64
        
        # Выходной слой
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)
        
    def forward(self, x, features):
        # Encoder для изображения
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        
        # Encoder для карты признаков
        f1 = self.feat_conv1(features)
        f2 = self.feat_down1(f1)
        f3 = self.feat_down2(f2)
        f4 = self.feat_down3(f3)  # Теперь у нас есть f4, размерность которого соответствует x4
        
        combined = torch.cat([x4, f4], dim=1) 
        
        x5 = self.bottleneck(combined)
        
        x = self.up1(x5)
        x = torch.cat([x, x4], dim=1)
        x = self.conv1(x)
        
        x = self.up2(x)
        x = torch.cat([x, x3], dim=1)
        x = self.conv2(x)
        
        x = self.up3(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv3(x)
        
        x = self.up4(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv4(x)
        
        x = self.outc(x)
        return torch.sigmoid(x)

def dice_loss(pred, target):
    smooth = 1.0
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    return 1 - (2.0 * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)

def iou_score(pred, target):
    pred = (pred > 0.5).float()
    target = target.float()
    
    intersection = (pred * target).sum().item()
    union = (pred + target).clamp(0, 1).sum().item()
    
    if union < 1e-6:
        return 0
    return intersection / union

def calculate_iou(true_mask, pred_mask):
    true_mask = true_mask.astype(bool)
    pred_mask = pred_mask.astype(bool)
    
    intersection = np.logical_and(true_mask, pred_mask)
    union = np.logical_or(true_mask, pred_mask)
    
    if np.sum(union) > 0:
        iou = np.sum(intersection) / np.sum(union)
    else:
        iou = 0.0
    
    return iou

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=20, device='cuda', is_enhanced=False):
    model.to(device)
    best_val_iou = 0.0
    history = {'train_loss': [], 'val_loss': [], 'train_iou': [], 'val_iou': []}
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_iou = 0.0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} (Train)"):
            if is_enhanced:
                images = batch['image'].to(device)
                features = batch['features'].to(device)
                masks = batch['mask'].to(device).unsqueeze(1)
                
                optimizer.zero_grad()
                outputs = model(images, features)
            else:
                images = batch['image'].to(device)
                masks = batch['mask'].to(device).unsqueeze(1)
                
                optimizer.zero_grad()
                outputs = model(images)
            
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_iou += iou_score(outputs, masks)
            
        train_loss /= len(train_loader)
        train_iou /= len(train_loader)
        
        model.eval()
        val_loss = 0.0
        val_iou = 0.0
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} (Val)"):
                if is_enhanced:
                    images = batch['image'].to(device)
                    features = batch['features'].to(device)
                    masks = batch['mask'].to(device).unsqueeze(1)
                    
                    outputs = model(images, features)
                else:
                    images = batch['image'].to(device)
                    masks = batch['mask'].to(device).unsqueeze(1)
                    
                    outputs = model(images)
                
                loss = criterion(outputs, masks)
                val_loss += loss.item()
                val_iou += iou_score(outputs, masks)
                
        val_loss /= len(val_loader)
        val_iou /= len(val_loader)
        
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_iou'].append(train_iou)
        history['val_iou'].append(val_iou)
        
        if val_iou > best_val_iou:
            best_val_iou = val_iou
            model_name = 'models/enhanced_unet.pth' if is_enhanced else 'models/standard_unet.pth'
            torch.save(model.state_dict(), model_name)
            print(f"Saved best model with IoU score: {val_iou:.4f}")
    
    return model, history

def post_process_masks(pred_masks, true_masks=None, lower_percentile=85, iterations=2):
    processed_masks = []
    
    for i, pred_mask in enumerate(pred_masks):

        binary_mask = binary_dilation(pred_mask, iterations=iterations)
        binary_mask = binary_closing(binary_mask, iterations=iterations)
        binary_mask = binary_fill_holes(binary_mask)
        
        labeled_mask, num_features = measure.label(binary_mask, return_num=True)
        if num_features > 1:
            regions = measure.regionprops(labeled_mask)
            
            if true_masks is not None:
                true_mask = true_masks[i]
                if np.any(true_mask):
                    true_regions = measure.regionprops(true_mask.astype(int))
                    if true_regions:
                        true_centroid = true_regions[0].centroid
                        closest_region = min(regions, 
                                           key=lambda r: ((r.centroid[0] - true_centroid[0])**2 + 
                                                         (r.centroid[1] - true_centroid[1])**2))
                        binary_mask = labeled_mask == closest_region.label
                    else:
                        largest_region = max(regions, key=lambda r: r.area)
                        binary_mask = labeled_mask == largest_region.label
                else:
                    largest_region = max(regions, key=lambda r: r.area)
                    binary_mask = labeled_mask == largest_region.label
            else:
                largest_region = max(regions, key=lambda r: r.area)
                binary_mask = labeled_mask == largest_region.label
        
        processed_masks.append(binary_mask)
    
    return np.array(processed_masks)

def evaluate_thresholds_and_plot(pred_probs, true_masks, percentile_range=(80, 99), num_steps=10, 
                                 model_name="Model", with_postprocessing=True):
    """
    Оптимизирует пороги для предсказаний вероятностей 
    """
    percentiles = np.linspace(percentile_range[0], percentile_range[1], num_steps)
    mean_ious = []
    post_mean_ious = []
    
    best_mean_iou = -1
    best_percentile = None
    best_masks = None
    best_image_ious = None
    
    best_post_mean_iou = -1
    best_post_percentile = None
    best_post_masks = None
    best_post_image_ious = None
    
    for percentile in percentiles:
        current_masks = []
        for pred_prob in pred_probs:
            threshold = np.percentile(pred_prob, percentile)
            binary_mask = pred_prob > threshold
            current_masks.append(binary_mask)
        
        current_ious = [calculate_iou(true, pred) for true, pred in zip(true_masks, current_masks)]
        current_mean_iou = np.mean(current_ious)
        
        mean_ious.append(current_mean_iou)
        
        if current_mean_iou > best_mean_iou:
            best_mean_iou = current_mean_iou
            best_percentile = percentile
            best_masks = current_masks
            best_image_ious = current_ious
        
        if with_postprocessing:
            post_masks = post_process_masks(current_masks, true_masks, lower_percentile=percentile)
            post_ious = [calculate_iou(true, pred) for true, pred in zip(true_masks, post_masks)]
            post_mean_iou = np.mean(post_ious)
            
            post_mean_ious.append(post_mean_iou)
            
            if post_mean_iou > best_post_mean_iou:
                best_post_mean_iou = post_mean_iou
                best_post_percentile = percentile
                best_post_masks = post_masks
                best_post_image_ious = post_ious
        
        print(f"Percentile {percentile:.1f}%: Mean IoU = {current_mean_iou:.4f}" + 
              (f", Post-processed IoU = {post_mean_iou:.4f}" if with_postprocessing else ""))
    
    plt.figure(figsize=(12, 6))
    plt.plot(percentiles, mean_ious, 'o-', color='blue', label='Original')
    
    if with_postprocessing:
        plt.plot(percentiles, post_mean_ious, 'o-', color='green', label='Post-processed')
        
        plt.axvline(x=best_percentile, color='blue', linestyle='--', 
                    label=f'Best original: {best_percentile:.1f}%')
        plt.axvline(x=best_post_percentile, color='green', linestyle='--', 
                    label=f'Best post-processed: {best_post_percentile:.1f}%')
    else:
        plt.axvline(x=best_percentile, color='red', linestyle='--', 
                    label=f'Best percentile: {best_percentile:.1f}%')
    
    plt.xlabel('Percentile threshold')
    plt.ylabel('Mean IoU')
    plt.title(f'IoU vs Threshold Percentile ({model_name})')
    plt.grid(True)
    plt.legend()
    plt.savefig(f"iou_vs_percentile_{model_name.replace(' ', '_')}.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    plt.figure(figsize=(12, 6))
    
    if with_postprocessing and best_post_mean_iou > best_mean_iou:
        plt.bar(range(len(best_post_image_ious)), best_post_image_ious, color='lightgreen')
        plt.axhline(y=best_post_mean_iou, color='green', linestyle='-', 
                    label=f'Mean Post-processed IoU: {best_post_mean_iou:.4f}')
        plt.title(f'IoU for each image with Post-processing ({model_name})')
        
        plt.bar(range(len(best_image_ious)), best_image_ious, color='lightblue', alpha=0.5)
        plt.axhline(y=best_mean_iou, color='blue', linestyle='--', 
                    label=f'Mean Original IoU: {best_mean_iou:.4f}')
    else:
        plt.bar(range(len(best_image_ious)), best_image_ious, color='skyblue')
        plt.axhline(y=best_mean_iou, color='blue', linestyle='-', 
                    label=f'Mean IoU: {best_mean_iou:.4f}')
        plt.title(f'IoU for each image ({model_name})')
    
    plt.xlabel('Image index')
    plt.ylabel('IoU')
    plt.xticks(range(len(best_image_ious)))
    plt.grid(True, axis='y')
    plt.legend()
    plt.savefig(f"iou_per_image_{model_name.replace(' ', '_')}.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    if with_postprocessing and best_post_mean_iou > best_mean_iou:
        return best_post_percentile, best_post_mean_iou, best_post_masks, best_post_image_ious
    else:
        return best_percentile, best_mean_iou, best_masks, best_image_ious

def visualize_comparison(tumor_images, true_masks, standard_masks, enhanced_masks, 
                         standard_ious, enhanced_ious):
    n_images = len(tumor_images)
    
    plt.figure(figsize=(15, 5*n_images))
    
    for i in range(n_images):
        # Оригинальное изображение
        plt.subplot(n_images, 4, i*4 + 1)
        plt.imshow(tumor_images[i], cmap='gray')
        plt.title(f"Исходное изображение {i+1}")
        plt.axis('off')
        
        # Истинная маска
        plt.subplot(n_images, 4, i*4 + 2)
        plt.imshow(tumor_images[i], cmap='gray')
        plt.imshow(true_masks[i], alpha=0.5, cmap='hot')
        plt.title(f"Истинная маска {i+1}")
        plt.axis('off')
        
        # Стандартная модель
        plt.subplot(n_images, 4, i*4 + 3)
        plt.imshow(tumor_images[i], cmap='gray')
        plt.imshow(standard_masks[i], alpha=0.5, cmap='hot')
        plt.title(f"Стандартная U-Net (IoU: {standard_ious[i]:.4f})")
        plt.axis('off')
        
        # Расширенная модель
        plt.subplot(n_images, 4, i*4 + 4)
        plt.imshow(tumor_images[i], cmap='gray')
        plt.imshow(enhanced_masks[i], alpha=0.5, cmap='hot')
        plt.title(f"U-Net + Геом/Топол (IoU: {enhanced_ious[i]:.4f})")
        plt.axis('off')
    
    plt.tight_layout()
    plt.savefig("comparison_standard_vs_enhanced.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    mean_standard_iou = np.mean(standard_ious)
    mean_enhanced_iou = np.mean(enhanced_ious)
    
    improvement = (mean_enhanced_iou - mean_standard_iou) / mean_standard_iou * 100
    
    plt.figure(figsize=(8, 6))
    plt.bar(['Стандартная U-Net', 'U-Net + Геом/Топол'], 
            [mean_standard_iou, mean_enhanced_iou], 
            color=['royalblue', 'forestgreen'])
    plt.ylabel('Средний IoU')
    plt.title(f'Сравнение производительности моделей\nУлучшение: {improvement:.2f}%')
    
    for i, v in enumerate([mean_standard_iou, mean_enhanced_iou]):
        plt.text(i, v + 0.01, f'{v:.4f}', ha='center')
    
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.savefig("iou_comparison_bar.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    plt.figure(figsize=(10, 6))
    
    iou_diffs = [enhanced - standard for enhanced, standard in zip(enhanced_ious, standard_ious)]
    colors = ['green' if diff > 0 else 'red' for diff in iou_diffs]
    
    plt.bar(range(len(iou_diffs)), iou_diffs, color=colors)
    plt.axhline(y=0, color='black', linestyle='-', linewidth=1)
    
    plt.xlabel('Индекс изображения')
    plt.ylabel('Прирост IoU (Enhanced - Standard)')
    plt.title('Прирост IoU при использовании топологических/геометрических признаков')
    plt.xticks(range(len(iou_diffs)))
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    for i, v in enumerate(iou_diffs):
        plt.text(i, v + 0.01 if v >= 0 else v - 0.03, f'{v:.4f}', ha='center')
    
    plt.savefig("iou_improvement_per_image.png", dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

num_epochs = 25

class BrainTumorDataset(Dataset):
    def __init__(self, images, masks, transform=None):
        self.images = images
        self.masks = masks
        self.transform = transform
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        mask = self.masks[idx]
        
        if len(image.shape) == 2:
            image = np.stack([image] * 3, axis=2)
            
        image = image.astype(np.float32) / 255.0
        
        image = torch.from_numpy(image.transpose(2, 0, 1)).float()
        mask = torch.from_numpy(mask.astype(np.float32))
        
        return {'image': image, 'mask': mask}

class EnhancedBrainTumorDataset(Dataset):
    def __init__(self, images, feature_maps, masks, transform=None):
        self.images = images
        self.feature_maps = feature_maps
        self.masks = masks
        self.transform = transform
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        feature_map = self.feature_maps[idx]
        mask = self.masks[idx]
        
        if len(image.shape) == 2:
            image = np.stack([image] * 3, axis=2)
            
        image = image.astype(np.float32) / 255.0
        feature_map = feature_map.astype(np.float32)
        
        image = torch.from_numpy(image.transpose(2, 0, 1)).float()
        feature_map = torch.from_numpy(feature_map).float().unsqueeze(0)  
        mask = torch.from_numpy(mask.astype(np.float32))
        
        return {'image': image, 'features': feature_map, 'mask': mask}

if __name__ == "__main__":
    base_path = "/Users/dasidorov03/PycharmProjects/dploma_4_course/RSNA-ASNR-MICCAI/brain_tumor_custom"
    tumor_images, true_masks = load_custom_dataset(base_path, max_samples=20) 
    
    print(f"Loaded {len(tumor_images)} images")
    
    print("Computing geometric/topological features...")
    feature_maps = compute_fiber_bundle_features(
        tumor_images,
        patch_size=6,
        stride=5,
        n_neighbors=5
    )
    
    X_train, X_test, y_train, y_test, f_train, f_test = train_test_split(
        tumor_images, true_masks, feature_maps, test_size=0.5, random_state=SEED
    )
    
    X_train, X_val, y_train, y_val, f_train, f_val = train_test_split(
        X_train, y_train, f_train, test_size=0.5, random_state=SEED
    )
    
    print(f"Training: {len(X_train)}, Validation: {len(X_val)}, Test: {len(X_test)}")
    
    train_dataset = BrainTumorDataset(X_train, y_train)
    val_dataset = BrainTumorDataset(X_val, y_val)
    test_dataset = BrainTumorDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=4)
    test_loader = DataLoader(test_dataset, batch_size=1)
    
    enhanced_train_dataset = EnhancedBrainTumorDataset(X_train, f_train, y_train)
    enhanced_val_dataset = EnhancedBrainTumorDataset(X_val, f_val, y_val)
    enhanced_test_dataset = EnhancedBrainTumorDataset(X_test, f_test, y_test)
    
    enhanced_train_loader = DataLoader(enhanced_train_dataset, batch_size=4, shuffle=True)
    enhanced_val_loader = DataLoader(enhanced_val_dataset, batch_size=4)
    enhanced_test_loader = DataLoader(enhanced_test_dataset, batch_size=1)
    
    print("\n=== Training Standard U-Net ===")
    lr = 0.01
    standard_model = UNet(in_channels=3, out_channels=1)
    optimizer = optim.Adam(standard_model.parameters(), lr=lr)
    criterion = dice_loss
    
    standard_model, standard_history = train_model(
        standard_model, train_loader, val_loader, criterion, optimizer, 
        num_epochs=num_epochs, device=DEVICE, is_enhanced=False
    )
    
    print("\n=== Training Enhanced U-Net with Geometric/Topological Features ===")
    enhanced_model = EnhancedUNet(in_channels=3, feature_channels=1, out_channels=1)
    optimizer = optim.Adam(enhanced_model.parameters(), lr=lr)
    
    enhanced_model, enhanced_history = train_model(
        enhanced_model, enhanced_train_loader, enhanced_val_loader, criterion, optimizer, 
        num_epochs=num_epochs, device=DEVICE, is_enhanced=True
    )
    
    print("\n=== Evaluating Models ===")
    
    standard_predictions = []
    standard_model.eval()
    with torch.no_grad():
        for batch in test_loader:
            images = batch['image'].to(DEVICE)
            outputs = standard_model(images)
            standard_predictions.append(outputs.cpu().numpy()[0, 0])
    
    enhanced_predictions = []
    enhanced_model.eval()
    with torch.no_grad():
        for batch in enhanced_test_loader:
            images = batch['image'].to(DEVICE)
            features = batch['features'].to(DEVICE)
            outputs = enhanced_model(images, features)
            enhanced_predictions.append(outputs.cpu().numpy()[0, 0])
    
    print("\n=== Optimizing Thresholds for Standard Model ===")
    standard_best_percentile, standard_best_iou, standard_best_masks, standard_image_ious = evaluate_thresholds_and_plot(
        standard_predictions, y_test, percentile_range=(80, 99.8), num_steps=20, 
        model_name="Standard U-Net", with_postprocessing=True
    )
    
    print("\n=== Optimizing Thresholds for Enhanced Model ===")
    enhanced_best_percentile, enhanced_best_iou, enhanced_best_masks, enhanced_image_ious = evaluate_thresholds_and_plot(
        enhanced_predictions, y_test, percentile_range=(80, 99.8), num_steps=20, 
        model_name="Enhanced U-Net", with_postprocessing=True
    )
    
    print("\n=== Comparison of Standard and Enhanced Models ===")
    print(f"Standard U-Net - Best Mean IoU: {standard_best_iou:.4f}")
    print(f"Enhanced U-Net - Best Mean IoU: {enhanced_best_iou:.4f}")
    
    improvement = (enhanced_best_iou - standard_best_iou) / standard_best_iou * 100
    print(f"Improvement: {improvement:.2f}%")
    
    visualize_comparison(
        X_test, y_test, 
        standard_best_masks, enhanced_best_masks,
        standard_image_ious, enhanced_image_ious
    )
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(standard_history['train_loss'], 'b-', label='Standard Train')
    plt.plot(standard_history['val_loss'], 'b--', label='Standard Val')
    plt.plot(enhanced_history['train_loss'], 'g-', label='Enhanced Train')
    plt.plot(enhanced_history['val_loss'], 'g--', label='Enhanced Val')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(standard_history['train_iou'], 'b-', label='Standard Train')
    plt.plot(standard_history['val_iou'], 'b--', label='Standard Val')
    plt.plot(enhanced_history['train_iou'], 'g-', label='Enhanced Train')
    plt.plot(enhanced_history['val_iou'], 'g--', label='Enhanced Val')
    plt.xlabel('Epoch')
    plt.ylabel('IoU Score')
    plt.title('Training and Validation IoU')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig("training_history.png", dpi=300, bbox_inches='tight')
    plt.show()

Using device: cpu
Loaded 20 images
Computing geometric/topological features...


Computing features: 100%|██████████| 20/20 [00:28<00:00,  1.45s/it]


Training: 5, Validation: 5, Test: 10

=== Training Standard U-Net ===


Epoch 1/25 (Val): 100%|██████████| 2/2 [00:01<00:00,  1.38it/s]


Epoch 1/25 - Train Loss: 0.9788, Val Loss: 0.9988


...

Epoch 21/25 (Val): 100%|██████████| 2/2 [00:01<00:00,  1.07it/s]


Epoch 21/25 - Train Loss: 0.1470, Val Loss: 0.6520


Epoch 22/25 (Val): 100%|██████████| 2/2 [00:01<00:00,  1.05it/s]


Epoch 22/25 - Train Loss: 0.1275, Val Loss: 0.3335
Saved best model with IoU score: 0.5570


Epoch 23/25 (Val):   0%|          | 0/2 [00:00<?, ?it/s]